<a href="https://colab.research.google.com/github/nayelivergaram-lang/VeteriPet-PQRS-UdeA./blob/main/Abrir%20codigo_VETERIPET_UDEA_en%20colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# VETERIPET-PQRS-UDEA
# PROYECTO INTEGRADOR - ALGORITMIA Y PROGRAMACIÓN
# ============================================================

from pathlib import Path
import sys
import json
import re
from datetime import datetime, date, timedelta
from collections import Counter
import importlib
import shutil

# ============================================================
# 1. ESTRUCTURA DEL PROYECTO
# ============================================================

BASE = Path("/content/VeteriPet-PQRS-Udea")
SRC = BASE / "src"
DATA = BASE / "data"
DOCS = BASE / "docs"
IMAGES = BASE / "images"

for carpeta in [SRC, DATA, DOCS, IMAGES]:
    carpeta.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. VALIDACIONES.PY
# ============================================================

validaciones = r'''
import re
from datetime import datetime, date, timedelta


def solicitar_texto(mensaje, minimo=1, maximo=None, obligatorio=True):
    """Texto limpio, claro, seguro y validado."""

    while True:
        valor = input(mensaje).strip()

        if not valor and not obligatorio:
            return ""

        if not valor:
            print("El campo es obligatorio.")
            continue

        if len(valor) < minimo:
            print(f"Debe tener mínimo {minimo} caracteres.")
            continue

        if maximo is not None and len(valor) > maximo:
            print(f"Debe tener máximo {maximo} caracteres.")
            continue

        return valor


def solicitar_nombre():
    """Nombre completo, alfabético, correcto y obligatorio."""

    patron = r"^[A-Za-zÁÉÍÓÚáéíóúÑñÜü]+(?:[ '-][A-Za-zÁÉÍÓÚáéíóúÑñÜü]+)+$"

    while True:
        nombre = solicitar_texto("Nombre completo: ", 3, 100)

        if re.fullmatch(patron, nombre):
            return nombre

        print("Use letras y escriba nombre y apellido.")


def solicitar_opcion(mensaje, opciones):
    """Menú numerado, sencillo, amigable y controlado."""

    while True:
        print(f"\n{mensaje}")

        for i, opcion in enumerate(opciones, 1):
            print(f"{i}. {opcion}")

        entrada = input("Seleccione una opción: ").strip()

        if entrada.isdigit() and 1 <= int(entrada) <= len(opciones):
            return opciones[int(entrada) - 1]

        print("Opción inválida.")


def solicitar_documento():
    """Documento numérico, obligatorio y validado."""

    while True:
        numero = input("Número de documento: ").strip()

        if numero.isdigit() and 3 <= len(numero) <= 15:
            return numero

        print("Debe contener entre 3 y 15 dígitos.")


def solicitar_telefono():
    """Teléfono numérico, obligatorio y de diez dígitos."""

    while True:
        telefono = input("Teléfono de contacto: ").strip()

        if telefono.isdigit() and len(telefono) == 10:
            return telefono

        print("Debe contener exactamente 10 dígitos.")


def solicitar_correo():
    """Correo electrónico válido, estructurado y obligatorio."""

    patron = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

    while True:
        correo = input("Correo electrónico: ").strip()

        if len(correo) <= 254 and re.fullmatch(patron, correo):
            return correo

        print("Ingrese un correo válido.")


def solicitar_fecha():
    """Fecha válida, actual o pasada, nunca futura."""

    while True:
        texto = input("Fecha de radicación (DD/MM/AAAA): ").strip()

        try:
            fecha = datetime.strptime(texto, "%d/%m/%Y").date()

            if fecha <= date.today():
                return fecha.isoformat()

            print("La fecha no puede ser futura.")

        except ValueError:
            print("Formato inválido. Use DD/MM/AAAA.")


def calcular_fecha_maxima(fecha):
    """Fecha máxima automática, exacta y calculada a 30 días."""

    fecha = date.fromisoformat(fecha)

    return (fecha + timedelta(days=30)).isoformat()


def solicitar_descripcion():
    """Descripción amplia, detallada y obligatoria."""

    return solicitar_texto(
        "Descripción detallada: ",
        minimo=20,
        maximo=2000
    )


def solicitar_datos_solicitante():
    """Datos completos, ordenados y validados."""

    return {
        "nombre_completo": solicitar_nombre(),

        "tipo_documento": solicitar_opcion(
            "Tipo de documento",
            ["CC", "TI", "CE", "PP", "NIT"]
        ),

        "numero_documento": solicitar_documento(),

        "tipo_telefono": solicitar_opcion(
            "Tipo de teléfono",
            ["Celular", "Fijo", "Corporativo", "Otro"]
        ),

        "telefono_contacto": solicitar_telefono(),

        "correo_electronico": solicitar_correo(),

        "direccion": solicitar_texto(
            "Dirección (opcional): ",
            minimo=5,
            maximo=200,
            obligatorio=False
        )
    }
'''

(SRC / "validaciones.py").write_text(validaciones, encoding="utf-8")

# ============================================================
# 3. CLASE BASE
# ============================================================

pqrs_base = r'''
class c_pqrs:
    """Clase base, organizada, reutilizable y estructurada."""

    def __init__(self, datos):
        self.datos = datos

    def actualizar_estado(self, nuevo_estado):
        """Actualización secuencial, controlada y válida."""

        flujo = {
            "Registrada": ["En proceso"],
            "En proceso": ["Solucionada"],
            "Solucionada": []
        }

        actual = self.datos["estado_PQRS"]

        if nuevo_estado in flujo.get(actual, []):
            self.datos["estado_PQRS"] = nuevo_estado
            return True

        return False
'''

(SRC / "pqrs_base.py").write_text(pqrs_base, encoding="utf-8")

# ============================================================
# 4. CLASES ESPECÍFICAS
# ============================================================

clases = {
    "peticion.py": ("c_peticion", "Petición"),
    "queja.py": ("c_queja", "Queja"),
    "reclamo.py": ("c_reclamo", "Reclamo"),
    "sugerencia.py": ("c_sugerencia", "Sugerencia")
}

for archivo, (clase, tipo) in clases.items():

    contenido = f'''
from pqrs_base import c_pqrs


class {clase}(c_pqrs):
    """Clase {tipo}, independiente, especializada y organizada."""

    TIPO_SOLICITUD = "{tipo}"

    def __init__(self, datos):
        datos["tipo_solicitud"] = self.TIPO_SOLICITUD
        super().__init__(datos)
'''

    (SRC / archivo).write_text(contenido, encoding="utf-8")

# ============================================================
# 5. ARCHIVOS.PY
# ============================================================

archivos = r'''
import json
from pathlib import Path

DATA = Path(__file__).resolve().parent.parent / "data"

ARCHIVOS = {
    "Petición": DATA / "Peticion.txt",
    "Queja": DATA / "Queja.txt",
    "Reclamo": DATA / "Reclamo.txt",
    "Sugerencia": DATA / "Sugerencia.txt"
}


def preparar_archivos():
    """Crea los archivos independientes y persistentes."""

    DATA.mkdir(parents=True, exist_ok=True)

    for ruta in ARCHIVOS.values():
        ruta.touch(exist_ok=True)


def leer_registros(tipo):
    """Lee registros JSON, incluyendo archivos con saltos corregidos."""

    ruta = ARCHIVOS[tipo]

    contenido = ruta.read_text(encoding="utf-8")

    # Separa correctamente registros que fueron guardados
    # con el texto literal de salto de línea.
    contenido = contenido.replace("\\\\n", "\n")

    registros = []

    for linea in contenido.splitlines():

        linea = linea.strip()

        if not linea:
            continue

        try:
            registros.append(json.loads(linea))

        except json.JSONDecodeError:
            print("Advertencia: se encontró un registro inválido.")

    return registros


def guardar_registro(registro):
    """Guarda un registro completo, independiente y persistente."""

    ruta = ARCHIVOS[registro["tipo_solicitud"]]

    with open(ruta, "a", encoding="utf-8") as archivo:

        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )


def obtener_siguiente_id(tipo):
    """Genera un ID entero, consecutivo e independiente por archivo."""

    registros = leer_registros(tipo)

    ids = [registro["id"] for registro in registros]

    return max(ids, default=0) + 1


def todos_los_registros():
    """Consulta los cuatro archivos sin unir su almacenamiento."""

    registros = []

    for tipo in ARCHIVOS:
        registros.extend(leer_registros(tipo))

    return registros


def actualizar_registro(tipo, id_registro, nuevo):
    """Actualiza un registro localizado y conserva los demás."""

    registros = leer_registros(tipo)

    for i, registro in enumerate(registros):

        if registro["id"] == id_registro:

            registros[i] = nuevo

            with open(ARCHIVOS[tipo], "w", encoding="utf-8") as archivo:

                for item in registros:

                    archivo.write(
                        json.dumps(item, ensure_ascii=False) + "\n"
                    )

            return True

    return False
'''

(SRC / "archivos.py").write_text(archivos, encoding="utf-8")

# ============================================================
# 6. REPORTES.PY
# ============================================================

reportes = r'''
from datetime import date, datetime
from collections import Counter
from pathlib import Path


def promedio_dias_respuesta(registros):
    """Calcula el promedio entero de días de respuesta."""

    dias = []

    for registro in registros:

        if registro["estado_PQRS"] == "Solucionada":

            if registro.get("fecha_solucion") and registro.get("fecha_registro") :

                inicio = datetime.fromisoformat(
                    registro["fecha_registro"]
                ).date()

                fin = date.fromisoformat(
                    registro["fecha_solucion"]
                )

                dias.append(max(0, (fin - inicio).days))

    return round(sum(dias) / len(dias)) if dias else 0


def generar_estadisticas(registros):
    """Genera estadísticas relevantes mediante comprensiones de listas."""

    tipos = Counter(
        r["tipo_solicitud"] for r in registros
    )

    mascotas = Counter(
        r["tipo_mascota"] for r in registros
    )

    estados = Counter(
        r["estado_PQRS"] for r in registros
    )

    proximas = [
        r for r in registros
        if r["estado_PQRS"] != "Solucionada"
        and 0 <= (
            date.fromisoformat(r["fecha_maxima_respuesta"])
            - date.today()
        ).days <= 5
    ]

    antiguas = sorted(
        registros,
        key=lambda r: r["fecha_radicacion"]
    )[:5]

    return {
        "Cantidad por tipo": dict(tipos),

        "Cantidad por mascota": dict(mascotas),

        "Cantidad por estado": dict(estados),

        "Promedio entero de días de respuesta":
            promedio_dias_respuesta(registros),

        "PQRS próximas a vencer":
            len(proximas),

        "Cinco PQRS más antiguas":
            antiguas
    }


def mostrar_estadisticas(registros):
    """Muestra resultados estadísticos claros y organizados."""

    if not registros:
        print("\nNo existen registros.")
        return

    resultados = generar_estadisticas(registros)

    print("\n" + "=" * 70)
    print("ESTADÍSTICAS VETERIPET-PQRS-UDEA")
    print("=" * 70)

    for nombre, valor in resultados.items():

        if nombre == "Cinco PQRS más antiguas":

            print(f"\n{nombre}:")

            for r in valor:

                print(
                    f"ID {r['id']} | "
                    f"{r['tipo_solicitud']} | "
                    f"{r['fecha_radicacion']} | "
                    f"{r['estado_PQRS']}"
                )

        else:

            print(f"\n{nombre}: {valor}")

def exportar_estadisticas(registros):
    """Exporta las estadisticas a un archivo te texto"""

    if not registros:
        print("\nNo existen registros para exportar")
        return

    resultados= generar_estadisticas(registros)
    ruta = Path(__file__).resolve().parent.parent / "data" / "Estadisticas.txt"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", encoding="utf-8" ) as archivo:
        archivo.write("ESTADÍSTICAS VETERIPET-PQRS-UDEA\n")
        archivo.write("=" *70 + "\n")

        for nombre, valor in resultados.items():
            archivo.write(f"\n{nombre}:\n")

            if nombre == "Cinco PQRS más antiguas":

                for r in valor:
                    archivo.write(
                        f"ID {r['id']} | "
                        f"{r['tipo_solicitud']} | "
                        f"{r['fecha_radicacion']} | "
                        f"{r['estado_PQRS']}\n"
                    )

            else:
                archivo.write(f"{valor}\n")

    print(f"\nEstadísticas exportadas correctamente en {ruta}.")
'''

(SRC / "reportes.py").write_text(reportes, encoding="utf-8")

# ============================================================
# 7. RADICADO.PY
# ============================================================

radicado = r'''

from pathlib import Path

BASE = Path(__file__).resolve().parent.parent
RADICADOS = BASE / "radicados"

CARPETAS_RADICADOS = {
    "Petición": "Peticion",
    "Queja": "Queja",
    "Reclamo": "Reclamo",
    "Sugerencia": "Sugerencia"
}

ANCHO = 120


def linea(texto="", centro=False):
    """Crea una línea uniforme y de ancho fijo."""

    contenido = str(texto)[:116]

    if centro:
        contenido = contenido.center(116)
    else:
        contenido = contenido.ljust(116)

    return "| " + contenido + " |"


def imprimir_radicado(registro, ruta_salida=None):
    """Genera un comprobante ASCII de 120 caracteres."""

    if ruta_salida is None:
        tipo = registro["tipo_solicitud"]
        carpeta = RADICADOS / CARPETAS_RADICADOS[tipo]
        carpeta.mkdir(parents=True, exist_ok=True)

        ruta_salida = (
            carpeta
            / f"radicado_{CARPETAS_RADICADOS[tipo].lower()}_{registro['id']}.txt"
        )
    else:
        ruta_salida = Path(ruta_salida)
        ruta_salida.parent.mkdir(parents=True, exist_ok=True)

    direccion = registro.get("direccion") or "N/A"

    lineas = [

        "+" + "-" * 118 + "+",

        linea("VETERIPET-PQRS-UDEA", True),

        linea("COMPROBANTE DE RADICACIÓN", True),

        "+" + "-" * 118 + "+",

        linea("DATOS DEL RADICADO", True),

        linea(f"ID: {registro['id']}"),

        linea(f"Tipo: {registro['tipo_solicitud']}"),

        linea(
            f"Fecha de radicación: "
            f"{registro['fecha_radicacion']} "
            f"{registro.get('hora_radicacion', 'No registrada')}"
        ),

        linea(
            f"Estado: "
            f"{registro['estado_PQRS']}"
        ),

        linea(
            f"Fecha máxima: "
            f"{registro['fecha_maxima_respuesta']}"
        ),

        "+" + "-" * 118 + "+",

        linea("DATOS DEL SOLICITANTE", True),

        linea(
            f"Nombre: "
            f"{registro['nombre_completo']}"
        ),

        linea(
            f"Documento: "
            f"{registro['tipo_documento']} - "
            f"{registro['numero_documento']}"
        ),

        linea(
            f"Teléfono: "
            f"{registro['telefono_contacto']} "
            f"({registro['tipo_telefono']})"
        ),

        linea(
            f"Correo: "
            f"{registro['correo_electronico']}"
        ),

        linea(
            f"Dirección: "
            f"{direccion}"
        ),

        "+" + "-" * 118 + "+",

        linea("CLASIFICACIÓN", True),

        linea(
            f"Canal: "
            f"{registro['canal_recepcion']}"
        ),

        linea(
            f"Mascota: "
            f"{registro['tipo_mascota']}"
        ),

        linea(
            f"Campus: "
            f"{registro['campus_relacionado']}"
        ),

        linea(
            f"Asunto: "
            f"{registro['asunto_titulo']}"
        ),

        "+" + "-" * 118 + "+",

        linea(
            "Descripción detallada almacenada en el archivo de datos.",
            True
        ),

        linea(
            "VeteriPet-PQRS-Udea",
            True
        ),

        "+" + "-" * 118 + "+"
    ]

    with open(ruta_salida, "w", encoding="utf-8") as archivo:

        archivo.write("\n".join(lineas))

    print(f"\nRadicado generado: {ruta_salida}")

    return ruta_salida
'''

(SRC / "radicado.py").write_text(radicado, encoding="utf-8")

# ============================================================
# 8. MAIN.PY
# ============================================================

main = r'''
from zoneinfo import ZoneInfo
from datetime import date, datetime
from pathlib import Path
import sys

SRC = Path(__file__).resolve().parent

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from validaciones import (
    solicitar_datos_solicitante,
    solicitar_opcion,
    solicitar_texto,
    solicitar_descripcion,
    solicitar_fecha,
    calcular_fecha_maxima
)

from archivos import (
    preparar_archivos,
    obtener_siguiente_id,
    guardar_registro,
    todos_los_registros,
    leer_registros,
    actualizar_registro
)

from reportes import mostrar_estadisticas, exportar_estadisticas
from radicado import imprimir_radicado

from peticion import c_peticion
from queja import c_queja
from reclamo import c_reclamo
from sugerencia import c_sugerencia


TIPOS = [
    "Petición",
    "Queja",
    "Reclamo",
    "Sugerencia"
]

CAMPUS = [
    "Campus Medellín - Ciudad Universitaria",
    "Campus Medellín - Ciudadela Robledo",
    "Campus en el Área de la salud",
    "Campus Medellín - Sede de Posgrado",
    "Campus Medellín - Edificio San Ignacio",
    "Campus Medellín - Antigua Escuela de Derecho",
    "Campus Medellín - Edificio Antioquia",
    "Campus Medellín - Casas Patrimoniales",
    "Campus Medellín - Edificio de Extensión"
]

CANALES = [
    "Presencial",
    "Correo electrónico",
    "Página web",
    "Teléfono",
    "Redes sociales",
    "Otro"
]

CLASES = {
    "Petición": c_peticion,
    "Queja": c_queja,
    "Reclamo": c_reclamo,
    "Sugerencia": c_sugerencia
}


def pausar():
    """Pausa sencilla, cómoda y controlada."""

    input("\nPresione ENTER para continuar...")


def registrar_pqrs():
    """Registra una PQRS completa, validada y persistente."""

    print("\n" + "=" * 70)
    print("REGISTRAR PQRS")
    print("=" * 70)

    datos = solicitar_datos_solicitante()

    tipo = solicitar_opcion(
        "Tipo de solicitud",
        TIPOS
    )

    fecha = solicitar_fecha()

    momento_registro = datetime.now(ZoneInfo("America/Bogota"))
    fecha_registro = momento_registro.isoformat(timespec="seconds")
    hora_radicacion = momento_registro.strftime("%H:%M:%S")


    datos.update({

        "tipo_solicitud": tipo,

        "fecha_radicacion": fecha,
        "fecha_registro": fecha_registro,
        "hora_radicacion": hora_radicacion,

        "canal_recepcion": solicitar_opcion(
            "Canal de recepción",
            CANALES
        ),

        "asunto_titulo": solicitar_texto(
            "Asunto o título: ",
            5,
            150
        ),

        "descripcion_detallada": solicitar_descripcion(),

        "tipo_mascota": solicitar_opcion(
            "Tipo de mascota",
            ["Perro", "Gato"]
        ),

        "campus_relacionado": solicitar_opcion(
            "Campus relacionado",
            CAMPUS
        ),

        "fecha_maxima_respuesta":
            calcular_fecha_maxima(fecha),

        "estado_PQRS": "Registrada",

        "fecha_solucion": None
    })

    datos["id"] = obtener_siguiente_id(tipo)

    objeto = CLASES[tipo](datos)

    guardar_registro(objeto.datos)

    print(
        f"\nPQRS registrada correctamente."
        f" ID: {datos['id']}"
    )

    imprimir_radicado(datos)

    pausar()


def buscar_registro():
    """Busca un registro exacto por tipo e ID."""

    tipo = solicitar_opcion(
        "Seleccione el tipo de PQRS",
        TIPOS
    )

    entrada = input("Ingrese el ID: ").strip()

    if not entrada.isdigit():

        print("El ID debe ser numérico.")

        return None

    id_buscado = int(entrada)

    registros = leer_registros(tipo)

    for registro in registros:

        if registro["id"] == id_buscado:

            return registro

    print("No se encontró el registro.")

    return None


def consultar_pqrs():
    """Consulta las PQRS activas en orden de antigüedad."""

    registros = todos_los_registros()

    activas = [
        r for r in registros
        if r["estado_PQRS"] != "Solucionada"
    ]

    print("\n" + "=" * 70)
    print("CONSULTAR PQRS ACTIVAS")
    print("=" * 70)

    if not activas:

        print("No hay PQRS activas.")

    else:

        for r in sorted(
            activas,
            key=lambda x: x["fecha_radicacion"]
        ):

            print(
                f"ID: {r['id']} | "
                f"Tipo: {r['tipo_solicitud']} | "
                f"Nombre: {r['nombre_completo']} | "
                f"Estado: {r['estado_PQRS']} | "
                f"Vence: {r['fecha_maxima_respuesta']}"
            )

    pausar()


def cambiar_estado():
    """Actualiza el estado respetando el flujo establecido."""

    registro = buscar_registro()

    if not registro:

        pausar()

        return

    flujo = {

        "Registrada": ["En proceso"],

        "En proceso": ["Solucionada"],

        "Solucionada": []
    }

    actual = registro["estado_PQRS"]

    if not flujo[actual]:

        print("El registro ya está solucionado.")

        pausar()

        return

    nuevo = solicitar_opcion(
        f"Estado actual: {actual}",
        flujo[actual]
    )

    registro["estado_PQRS"] = nuevo

    if nuevo == "Solucionada":

        registro["fecha_solucion"] = date.today().isoformat()

    actualizar_registro(
        registro["tipo_solicitud"],
        registro["id"],
        registro
    )

    print("Estado actualizado correctamente.")

    pausar()


def imprimir_consulta():
    """Genera un radicado actualizado y exportable."""

    registro = buscar_registro()

    if registro:

        imprimir_radicado(registro)

    pausar()


def menu():
    """Menú principal, amigable, repetitivo y funcional."""

    preparar_archivos()

    while True:

        print("\n" + "=" * 70)
        print("VETERIPET-PQRS-UDEA")
        print("=" * 70)

        print("1. Registrar PQRS")
        print("2. Consultar PQRS")
        print("3. Registrar cambio en PQRS")
        print("4. Estadísticas")
        print("5. Imprimir radicado")
        print("6. Exportar estadísticas")
        print("7. Salir")

        opcion = input("\nSeleccione una opción: ").strip()

        if opcion == "1":

            registrar_pqrs()

        elif opcion == "2":

            consultar_pqrs()

        elif opcion == "3":

            cambiar_estado()

        elif opcion == "4":

            mostrar_estadisticas(
                todos_los_registros()
            )

            pausar()

        elif opcion == "5":

            imprimir_consulta()

        elif opcion == "6":
            exportar_estadisticas(todos_los_registros())

            pausar()

        elif opcion == "7":

            print(
                "\nGracias por utilizar "
                "VeteriPet-PQRS-Udea."
            )

            break

        else:

            print("Opción inválida.")


if __name__ == "__main__":

    menu()
'''

(SRC / "main.py").write_text(main, encoding="utf-8")

# ============================================================
# 9. ARCHIVOS TXT Y DOCUMENTACIÓN
# ============================================================

for archivo in [
    "Peticion.txt",
    "Queja.txt",
    "Reclamo.txt",
    "Sugerencia.txt"
]:

    (DATA / archivo).touch(exist_ok=True)

if not (DOCS / "manual_usuario.md").exists():
    (DOCS / "manual_usuario.md").write_text(
    "# Manual de Usuario\n\n"
    "## VeteriPet-PQRS-Udea\n\n"
    "Sistema de gestión de PQRS.\n",
    encoding="utf-8"
)

if not (BASE / "README.md").exists():
   (BASE / "README.md").write_text(
    "# VeteriPet-PQRS-Udea\n\n"
    "Gestor de PQRS para perros y gatos.\n",
    encoding="utf-8"
)

# ============================================================
# 10. VERIFICACIÓN DE SINTAXIS
# ============================================================

import py_compile

for archivo in SRC.glob("*.py"):

    py_compile.compile(
        str(archivo),
        doraise=True
    )

print("=" * 60)
print("VERIFICACIÓN DE SINTAXIS COMPLETADA")
print("Los archivos Python tienen sintaxis válida.")
print("=" * 60)


%run /content/VeteriPet-PQRS-Udea/src/main.py

VERIFICACIÓN DE SINTAXIS COMPLETADA
Los archivos Python tienen sintaxis válida.

VETERIPET-PQRS-UDEA
1. Registrar PQRS
2. Consultar PQRS
3. Registrar cambio en PQRS
4. Estadísticas
5. Imprimir radicado
6. Exportar estadísticas
7. Salir

Seleccione una opción: 1

REGISTRAR PQRS
Nombre completo: Laura Marcela Gómez Ruiz

Tipo de documento
1. CC
2. TI
3. CE
4. PP
5. NIT
Seleccione una opción: 1
Número de documento: 1000000001

Tipo de teléfono
1. Celular
2. Fijo
3. Corporativo
4. Otro
Seleccione una opción: 1
Teléfono de contacto: 3000000001
Correo electrónico: laura.gomez@example.com
Dirección (opcional): Calle 10 # 20-30

Tipo de solicitud
1. Petición
2. Queja
3. Reclamo
4. Sugerencia
Seleccione una opción: 1
Fecha de radicación (DD/MM/AAAA): 03/11/2026

Canal de recepción
1. Presencial
2. Correo electrónico
3. Página web
4. Teléfono
5. Redes sociales
6. Otro
Seleccione una opción: 3
Asunto o título: Solicitud de jornada de vacunación
Descripción detallada: 	Solicito información sobre l